# Candidate Screening Demonstration

## Scientific objective
Run single-SMILES and batch inference and return one calibrated, uncertainty-aware, AD/OOD-controlled recommendation per endpoint.

## Inputs
- `models/calibrated/*.joblib`
- `app/example_inputs.csv`

## Expected outputs
- `results/predictions/candidate_screening_demo.csv`
- endpoint-specific table shown in notebook

## Dependencies
joblib, RDKit

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
Demonstration molecules are not evidence of clinical safety. Predictions can abstain and require experimental confirmation.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Interpretations are model-behavior summaries. No endpoint should be collapsed into an overall safe/unsafe label.

## Next notebook
[24_reproducibility_audit.ipynb](./24_reproducibility_audit.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'smoke', 'seed': 20260723}


In [2]:
from toxicity_screening.inference import load_bundles, predict_smiles
bundles=load_bundles(ROOT/"models/calibrated")
examples=pd.read_csv(ROOT/"app/example_inputs.csv")
outputs=[]
for row in examples.itertuples(index=False): outputs.append(predict_smiles(row.smiles,bundles,str(row.molecule_id)))
result=pd.concat(outputs,ignore_index=True); result.to_csv(ROOT/"results/predictions/candidate_screening_demo.csv",index=False)
expected={"molecule_id","original_smiles","standardized_smiles","endpoint","predicted_class","raw_probability","calibrated_probability","uncertainty","applicability_domain","nearest_training_similarity","scaffold_novelty","ood_warning","abstention_status","interpretation","recommendation"}
assert expected.issubset(result.columns); assert result.groupby("molecule_id").endpoint.nunique().eq(len(bundles)).all()
display(result)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_8620\1274677326.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result=pd.concat(outputs,ignore_index=True); result.to_csv(ROOT/"results/predictions/candidate_screening_demo.csv",index=False)


,molecule_id,original_smiles,standardized_smiles,endpoint,predicted_class,raw_probability,calibrated_probability,uncertainty,applicability_domain,nearest_training_similarity,scaffold_novelty,ood_warning,abstention_status,interpretation,recommendation
0,aspirin,CC(=O)OC1=CC=CC=C1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,ames_mutagenicity,0.0,0.098700,0.041667,0.073691,inside,1.000000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
1,aspirin,CC(=O)OC1=CC=CC=C1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,herg_blockade,0.0,0.094605,0.103093,0.066334,inside,1.000000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
2,aspirin,CC(=O)OC1=CC=CC=C1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,SR-ARE,0.0,0.036493,0.000000,0.024011,inside,1.000000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
3,aspirin,CC(=O)OC1=CC=CC=C1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,SR-ATAD5,0.0,0.003624,0.000000,0.004946,inside,1.000000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
4,aspirin,CC(=O)OC1=CC=CC=C1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,SR-MMP,0.0,0.030104,0.035088,0.024063,inside,1.000000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
5,aspirin,CC(=O)OC1=CC=CC=C1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,SR-p53,0.0,0.104927,0.069767,0.096287,inside,1.000000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
6,caffeine,CN1C(=O)N(C)c2ncn(C)c2C1=O,Cn1c(=O)c2c(ncn2C)n(C)c1=O,ames_mutagenicity,0.0,0.098322,0.041667,0.066387,inside,1.000000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
7,caffeine,CN1C(=O)N(C)c2ncn(C)c2C1=O,Cn1c(=O)c2c(ncn2C)n(C)c1=O,herg_blockade,NaN,0.312077,0.405263,0.385915,inside,0.529412,False,True,abstain,"{""ood_reasons"": [""high_uncertainty""]}",Abstain because the molecule is outside the ap...
8,caffeine,CN1C(=O)N(C)c2ncn(C)c2C1=O,Cn1c(=O)c2c(ncn2C)n(C)c1=O,SR-ARE,0.0,0.024569,0.000000,0.021569,inside,0.600000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...
9,caffeine,CN1C(=O)N(C)c2ncn(C)c2C1=O,Cn1c(=O)c2c(ncn2C)n(C)c1=O,SR-ATAD5,0.0,0.003353,0.000000,0.004168,inside,1.000000,False,False,supported,"{""ood_reasons"": []}",Low predicted concern; experimental confirmati...


### Completion gate
Confirm that the declared artifacts exist before continuing to `24_reproducibility_audit.ipynb`.